# Eindwerk Data Science — Medische Risicoanalyse

**Auteur:** Eindwerk Jaar 1  
**Dataset:** Medische dienst — risico's en pathologieën van werknemers  
**Taal data:** Frans  

---

## Overzicht

In dit notebook analyseren we een dataset van een medische dienst met meer dan 1 miljoen rijen.  
We doorlopen de volgende stappen:

1. Data inlezen
2. Data opkuisen
3. Algemene informatie
4. Visualisaties

## 0. Imports en configuratie

In [ ]:
import sys
import os

# Voeg de rootmap toe aan het pad zodat we Lib kunnen importeren
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import plotly.io as pio

from Lib.data_laden import laad_alle_data
from Lib.data_opkuisen import kuise_data_op, filter_laatste_n_jaar
from Lib import visualisaties as vis

# Plotly-weergave instellen voor Jupyter
pio.renderers.default = 'notebook'

# Pad naar de datamap
DATA_MAP = '../Data'

print('Imports geladen.')

---

## 1. Data inlezen

In [ ]:
# Laad alle CSV-bestanden en voeg opzoektabellen samen
print('Data inlezen... (dit kan even duren bij 1M+ rijen)')
df_ruw = laad_alle_data(DATA_MAP)
print(f'Data ingelezen: {len(df_ruw):,} rijen, {df_ruw.shape[1]} kolommen')

In [ ]:
# Eerste rijen bekijken
df_ruw.head()

---

## 2. Data opkuisen

In [ ]:
# Opkuisen: verwijder ongeldige risico-codes, strip ICD-10 spaties,
# parseer datums, voeg leeftijdsgroep en seizoen toe
rijen_voor = len(df_ruw)

df = kuise_data_op(df_ruw)

rijen_na = len(df)
verwijderd = rijen_voor - rijen_na

print(f'Rijen vóór opkuisen : {rijen_voor:,}')
print(f'Rijen ná opkuisen   : {rijen_na:,}')
print(f'Verwijderde rijen   : {verwijderd:,} ({verwijderd / rijen_voor * 100:.2f}%)')

In [ ]:
# Controleer dat de te verwijderen risico-codes niet meer aanwezig zijn
from Lib.data_opkuisen import TE_VERWIJDEREN_RISICOS

nog_aanwezig = df[df['risk_code'].isin(TE_VERWIJDEREN_RISICOS)]
print(f'Rijen met ongeldige risico-codes na opkuisen: {len(nog_aanwezig)}')
assert len(nog_aanwezig) == 0, 'Er zijn nog ongeldige risico-codes aanwezig!'

---

## 3. Algemene informatie

In [ ]:
# Structuur en datatypes
df.info(memory_usage='deep')

In [ ]:
# Statistische samenvatting
df.describe(include='all')

In [ ]:
# Ontbrekende waarden per kolom
ontbrekend = df.isnull().sum().rename('Ontbrekende waarden')
pct = (ontbrekend / len(df) * 100).rename('Percentage (%)')
overzicht = pd.concat([ontbrekend, pct], axis=1)
overzicht[overzicht['Ontbrekende waarden'] > 0].sort_values('Ontbrekende waarden', ascending=False)

In [ ]:
# Unieke waarden per categorische kolom
print('Unieke waarden per kolom:')
for kolom in ['risk_code', 'pathology_icd10code', 'nace_code', 'employee_sex', 'employment_functioncode', 'seizoen', 'leeftijdsgroep']:
    if kolom in df.columns:
        n = df[kolom].nunique()
        print(f'  {kolom:35s}: {n:,}')

In [ ]:
# Waarschuwing over employment_functioncode
pct_null_functie = df['employment_functioncode'].isnull().mean() * 100
print(f'Let op: {pct_null_functie:.1f}% van de rijen heeft geen functiecode.')
print('Plots op basis van functiecode zijn daardoor gebaseerd op een beperkt deel van de data.')

In [ ]:
# Spreiding van leeftijden
print('Leeftijdsstatistieken:')
print(df['employee_age_start_pathology'].describe())
print(f"\nOnrealistische leeftijden (< 0 of > 100): {((df['employee_age_start_pathology'] < 0) | (df['employee_age_start_pathology'] > 100)).sum():,} rijen")

---

## 4. Visualisaties

### 4.1 Meest voorkomende risico's, pathologieën en combinaties (alle jaren)

In [ ]:
# Top 10 meest voorkomende risico's
vis.plot_top10_risicos(df).show()

In [ ]:
# Top 10 meest voorkomende pathologieën
vis.plot_top10_pathologieen(df).show()

In [ ]:
# Top 10 meest voorkomende risico–pathologie combinaties
vis.plot_top10_risico_pathologie_combo(df).show()

### 4.2 Meest voorkomende risico's, pathologieën en combinaties (laatste 5 jaar)

In [ ]:
# Filter op de laatste 5 jaar
df_5j = filter_laatste_n_jaar(df, n=5)
print(f'Rijen in de laatste 5 jaar: {len(df_5j):,}')

In [ ]:
vis.plot_top10_risicos(df_5j, titel_suffix='Laatste 5 jaar').show()

In [ ]:
vis.plot_top10_pathologieen(df_5j, titel_suffix='Laatste 5 jaar').show()

In [ ]:
vis.plot_top10_risico_pathologie_combo(df_5j, titel_suffix='Laatste 5 jaar').show()

### 4.3 Demografie — pathologieën per geslacht en per leeftijdsgroep

In [ ]:
vis.plot_pathologie_per_geslacht(df).show()

In [ ]:
vis.plot_pathologie_per_leeftijdsgroep(df).show()

### 4.4 Pathologieën per seizoen

In [ ]:
vis.plot_pathologie_per_seizoen(df).show()

### 4.5 Per sector (NACE)

In [ ]:
vis.plot_risico_per_nace(df).show()

In [ ]:
vis.plot_pathologie_per_nace(df).show()

### 4.6 Per functiecode

> **Opmerking:** De kolom `employment_functioncode` is grotendeels leeg in de dataset.  
> De onderstaande plots zijn gebaseerd op een beperkt deel van de data.

In [ ]:
vis.plot_risico_per_functiecode(df).show()

In [ ]:
vis.plot_pathologie_per_functiecode(df).show()